<style>
table { margin-left: 0 !important; margin-right: auto !important; }
th, td { text-align: left !important; }
</style>

## 02-2 · Mathematics for Optimization: Vectors, Inner Products, Norms, Level Sets, and Gradients

**We want to predict how a small change in the cooling decision will change the score before evaluating every possible decision.**

Lecture 02-1 formulated the classroom problem with decision vector \(x=[u_{\mathrm{early}},u_{\mathrm{late}}]^{\mathsf T}\), responses \(y=\operatorname{Sim}(x)\), objective \(f\), and constraints \(g,h\). This lecture keeps that formulation and adds the geometry needed to reason about search directions.

The physical state transition \(F\), external inputs, fixed parameters, and requirement limits remain unchanged. Only the mathematical view of the decision space is new.


### 1 · Follow the 75-minute route

| Time | Activity | Question answered |
|:---:|:---|:---|
| 0–8 min | Opening prediction | What does one point in decision space mean physically? |
| 8–23 min | Vector geometry | How do coordinates represent early and late cooling? |
| 23–38 min | Inner-product diagnosis | Does a proposed step align with increase or decrease? |
| 38–52 min | Norm comparison | How large is the same decision change under different rules? |
| 52–67 min | Level-set and gradient reasoning | Which local direction changes the score fastest? |
| 67–75 min | Transfer and exit ticket | Which mathematical object answers each search question? |

The intervals total 75 minutes. At each stage, hold the classroom system fixed and change only the mathematical object under discussion.


### 2 · A vector records one complete decision

The two coordinates of the decision vector answer two physical questions:

> $\displaystyle x=\begin{bmatrix}x_1\\x_2\end{bmatrix}=\begin{bmatrix}u_{\mathrm{early}}\\u_{\mathrm{late}}\end{bmatrix}.$

The point \(x=[3,2]^{\mathsf T}\) means 3 cooling units for \(t=0,\ldots,5\) and 2 cooling units for \(t=6,\ldots,11\). The vector is chosen directly. The temperature path is a system state produced after that choice.

The shaded square below is the bound-constrained decision domain \(\mathcal X=[0,5]^2\). Compare the two blue coordinate components with the orange candidate they locate.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/02-2_mathematics_for_optimization/assets/01_decision_vector_geometry.svg" alt="Decision-space diagram showing early and late cooling coordinates, the allowed square domain, and the candidate vector from the origin to three early and two late cooling units" width="670" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

The code defines the carried-forward classroom system and recreates the vector diagram.


In [ ]:
import sys
from types import SimpleNamespace

import matplotlib
import numpy as np


def _pyplot(*, interactive=False):
    """Return pyplot, activating ipympl for interactive figures when available."""
    if interactive and sys.platform != "emscripten":
        try:
            matplotlib.use("widget", force=True)
        except (RuntimeError, ValueError):
            from matplotlib.backends import backend_registry

            backend_registry._clear()
            matplotlib.use("widget", force=True)
    import matplotlib.pyplot as plt

    return plt


# Horizon and initial state
TIME_STEPS = 12
INITIAL_TEMPERATURE = 27.0

# Fixed parameters
WEATHER_EXCHANGE = 0.12
OCCUPANT_HEAT = 0.012
COOLING_EFFECT = 0.45

# External inputs
OUTSIDE_TEMPERATURE = np.full(TIME_STEPS, 31.0)
OCCUPANTS = np.full(TIME_STEPS, 20.0)

# Decision domain and requirement limits
MIN_COOLING, MAX_COOLING = 0.0, 5.0
MIN_TEMPERATURE, MAX_TEMPERATURE = 20.0, 30.0
MAX_ENERGY = 60.0

# Stable visual roles
BLUE = "#2563EB"
TEAL = "#0F9D8A"
ORANGE = "#F28E2B"
PURPLE = "#7C3AED"
GRAY = "#A7AFB9"
INK = "#172033"
PAPER = "#F8FAFC"


def expand_decision(x):
    early_cooling, late_cooling = np.asarray(x, dtype=float)
    return np.r_[np.full(6, early_cooling), np.full(6, late_cooling)]


def simulation_model(x):
    cooling_schedule = expand_decision(x)
    temperatures = [INITIAL_TEMPERATURE]
    for outdoor, people, cooling in zip(
        OUTSIDE_TEMPERATURE, OCCUPANTS, cooling_schedule
    ):
        current = temperatures[-1]
        temperatures.append(
            current
            + WEATHER_EXCHANGE * (outdoor - current)
            + OCCUPANT_HEAT * people
            - COOLING_EFFECT * cooling
        )
    temperatures = np.asarray(temperatures)
    discomfort = np.sum(
        np.maximum(temperatures[1:] - 24.0, 0.0) ** 2
        + np.maximum(22.0 - temperatures[1:], 0.0) ** 2
    )
    energy = 0.5 * np.sum(cooling_schedule**2)
    return {
        "temperatures": temperatures,
        "discomfort": float(discomfort),
        "energy": float(energy),
    }


def score(x, energy_weight=1.0):
    response = simulation_model(x)
    return response["discomfort"] + energy_weight * response["energy"]


def is_feasible(x):
    x = np.asarray(x, dtype=float)
    response = simulation_model(x)
    temperatures = response["temperatures"][1:]
    return bool(
        np.all((MIN_COOLING <= x) & (x <= MAX_COOLING))
        and temperatures.min() >= MIN_TEMPERATURE
        and temperatures.max() <= MAX_TEMPERATURE
        and response["energy"] <= MAX_ENERGY
    )


def finite_difference_gradient(x, energy_weight=1.0, step=1e-3):
    x = np.asarray(x, dtype=float)
    gradient = np.empty_like(x)
    for index in range(len(x)):
        offset = np.zeros_like(x)
        offset[index] = step
        gradient[index] = (
            score(x + offset, energy_weight)
            - score(x - offset, energy_weight)
        ) / (2 * step)
    return gradient


def style_axis(axis):
    axis.set_facecolor(PAPER)
    axis.spines[["top", "right"]].set_visible(False)
    axis.grid(alpha=0.25, color=GRAY, linewidth=0.8)


def show_decision_vector(decision=(3.0, 2.0)):
    plt = _pyplot()
    early, late = map(float, decision)
    figure, axis = plt.subplots(figsize=(7.2, 5.6))
    axis.axvspan(0, 5, color=TEAL, alpha=0.08)
    axis.axhspan(0, 5, color=TEAL, alpha=0.08)
    axis.plot([early, early], [0, late], color=BLUE, linestyle=(0, (4, 4)), alpha=0.55)
    axis.plot([0, early], [late, late], color=BLUE, linestyle=(0, (4, 4)), alpha=0.55)
    axis.annotate(
        "", xy=(early, late), xytext=(0, 0),
        arrowprops=dict(arrowstyle="-|>", color=BLUE, linewidth=3, mutation_scale=18),
    )
    axis.scatter(early, late, s=150, color=ORANGE, edgecolor="white", linewidth=2.2, zorder=4)
    axis.text(early + 0.16, late + 0.18, r"$x=[3,2]^{\mathsf{T}}$", color=INK, fontsize=12, weight="bold")
    axis.text(early / 2, -0.34, r"$x_1=u_{\mathrm{early}}=3$", ha="center", color=BLUE, fontsize=10)
    axis.text(-0.38, late / 2, r"$x_2=u_{\mathrm{late}}=2$", va="center", ha="right", rotation=90, color=BLUE, fontsize=10)
    axis.text(4.82, 4.72, r"$\mathcal{X}=[0,5]^2$", ha="right", color=TEAL, fontsize=11, weight="bold")
    axis.set(
        xlim=(-0.55, 5.35), ylim=(-0.55, 5.35),
        xlabel=r"Early cooling $x_1$ (cooling units)",
        ylabel=r"Late cooling $x_2$ (cooling units)",
        title="One vector identifies one complete cooling decision",
        xticks=np.arange(0, 6), yticks=np.arange(0, 6),
    )
    style_axis(axis)
    axis.set_aspect("equal")
    figure.tight_layout()
    plt.show()
    plt.close(figure)
    return np.asarray(decision, dtype=float)


decision = show_decision_vector((3.0, 2.0))
print(f"The decision has {decision.size} coordinates and norm {np.linalg.norm(decision):.2f}.")


### 3 · An inner product measures alignment

For vectors \(a,b\in\mathbb R^p\), the inner product is

> $\displaystyle a^{\mathsf T}b=\sum_{i=1}^{p}a_i b_i=\lVert a\rVert_2\lVert b\rVert_2\cos\theta.$

It answers an alignment question. A positive value means the angle \(\theta\) is acute, zero means the vectors are perpendicular, and a negative value means they point generally in opposite directions.

In optimization, let \(a=\nabla J(x)\) and let \(b=d\) be a proposed decision step. Then \(\nabla J(x)^{\mathsf T}d\) predicts the first-order score change. In the figure, the orange step points against the blue gradient, so the inner product is negative.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/02-2_mathematics_for_optimization/assets/02_inner_product_directional_change.svg" alt="Geometric inner-product diagram showing a blue objective gradient, an orange proposed decision step, their obtuse angle, and the negative projection that predicts a score decrease" width="670" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

The calculation below uses the classroom score at \(x=[3,2]^{\mathsf T}\).


In [ ]:
def show_inner_product(decision=(3.0, 2.0), proposed_step=(1.0, -0.45)):
    plt = _pyplot()
    gradient = finite_difference_gradient(decision)
    gradient_direction = gradient / np.linalg.norm(gradient)
    step_direction = np.asarray(proposed_step, dtype=float)
    step_direction /= np.linalg.norm(step_direction)
    inner_product = float(gradient_direction @ step_direction)
    angle = np.degrees(np.arccos(np.clip(inner_product, -1.0, 1.0)))

    figure, axis = plt.subplots(figsize=(7.2, 5.6))
    theta = np.linspace(0, 2 * np.pi, 361)
    axis.fill(3.05 * np.cos(theta), 3.05 * np.sin(theta), color=PAPER, zorder=0)
    for vector, color, label in [
        (2.45 * gradient_direction, BLUE, r"gradient $\nabla J$"),
        (2.25 * step_direction, ORANGE, r"proposed step $d$"),
    ]:
        axis.annotate(
            "", xy=vector, xytext=(0, 0),
            arrowprops=dict(arrowstyle="-|>", color=color, linewidth=3, mutation_scale=18),
        )
        axis.text(*(vector * 1.08), label, color=color, fontsize=11, weight="bold", ha="center")

    gradient_angle = np.arctan2(gradient_direction[1], gradient_direction[0])
    step_angle = np.arctan2(step_direction[1], step_direction[0])
    arc_angles = np.linspace(step_angle, gradient_angle + 2 * np.pi, 120)
    arc_angles = arc_angles[arc_angles <= step_angle + np.pi]
    if len(arc_angles) < 2:
        arc_angles = np.linspace(gradient_angle, step_angle, 120)
    axis.plot(0.72 * np.cos(arc_angles), 0.72 * np.sin(arc_angles), color=INK, linewidth=1.6)
    axis.text(0.02, 0.92, rf"$\theta={angle:.0f}^\circ$", transform=axis.transAxes, color=INK, fontsize=11)
    axis.text(0.02, 0.84, rf"$\nabla J^{{\mathsf{{T}}}}d={inner_product:.2f}<0$", transform=axis.transAxes, color=TEAL, fontsize=12, weight="bold")
    axis.text(0.02, 0.77, "first-order prediction: score decreases", transform=axis.transAxes, color=TEAL, fontsize=10)
    axis.axhline(0, color=GRAY, linewidth=1)
    axis.axvline(0, color=GRAY, linewidth=1)
    axis.set(
        xlim=(-3.1, 3.1), ylim=(-3.1, 3.1),
        xlabel=r"Early-cooling direction $d_1$",
        ylabel=r"Late-cooling direction $d_2$",
        title="The inner product predicts local directional change",
    )
    style_axis(axis)
    axis.set_aspect("equal")
    figure.tight_layout()
    plt.show()
    plt.close(figure)
    return SimpleNamespace(gradient=gradient, direction=step_direction, inner_product=inner_product)


alignment = show_inner_product()
print(f"gradient = {alignment.gradient.round(3)}")
print(f"normalized inner product = {alignment.inner_product:.3f}")


### 4 · A norm defines the size of a decision change

A norm maps a vector to a nonnegative scalar. Three common norms are

> $\displaystyle \lVert d\rVert_1=\sum_i|d_i|,\qquad \lVert d\rVert_2=\sqrt{\sum_i d_i^2},\qquad \lVert d\rVert_\infty=\max_i|d_i|.$

The norm is an analyst choice. It does not change the physical outcome of a fixed cooling decision. It changes how the algorithm measures distance, step size, or proximity.

The next figure holds the same change \(d=[0.75,0.50]^{\mathsf T}\) fixed. Only the norm changes. Compare the orange point with each unit boundary.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/02-2_mathematics_for_optimization/assets/03_norm_unit_balls.svg" alt="Three coordinated panels comparing L1 diamond, L2 circle, and L-infinity square unit balls for the same orange cooling-decision change" width="820" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>


In [ ]:
def show_norm_comparison(change=(0.75, 0.50)):
    plt = _pyplot()
    change = np.asarray(change, dtype=float)
    angle = np.linspace(0, 2 * np.pi, 361)
    boundaries = [
        (np.r_[np.linspace(0, 1, 80), np.linspace(1, 0, 80), np.linspace(0, -1, 80), np.linspace(-1, 0, 80)],
         np.r_[np.linspace(1, 0, 80), np.linspace(0, -1, 80), np.linspace(-1, 0, 80), np.linspace(0, 1, 80)]),
        (np.cos(angle), np.sin(angle)),
        (np.array([-1, 1, 1, -1, -1]), np.array([-1, -1, 1, 1, -1])),
    ]
    norm_values = [np.linalg.norm(change, 1), np.linalg.norm(change, 2), np.linalg.norm(change, np.inf)]
    labels = [r"$\ell_1$ · total coordinate change", r"$\ell_2$ · straight-line change", r"$\ell_\infty$ · largest coordinate change"]

    figure, axes = plt.subplots(1, 3, figsize=(11.4, 4.8), sharex=True, sharey=True)
    for axis, boundary, value, label in zip(axes, boundaries, norm_values, labels):
        axis.fill(boundary[0], boundary[1], color=TEAL, alpha=0.12)
        axis.plot(boundary[0], boundary[1], color=BLUE, linewidth=2.4)
        axis.annotate(
            "", xy=change, xytext=(0, 0),
            arrowprops=dict(arrowstyle="-|>", color=ORANGE, linewidth=2.5, mutation_scale=15),
        )
        axis.scatter(*change, s=85, color=ORANGE, edgecolor="white", linewidth=1.5, zorder=4)
        axis.text(0.04, 0.93, rf"$\|d\|={value:.2f}$", transform=axis.transAxes, color=INK, fontsize=11, weight="bold")
        axis.set(title=label, xlim=(-1.3, 1.3), ylim=(-1.3, 1.3), xticks=[-1, 0, 1], yticks=[-1, 0, 1])
        style_axis(axis)
        axis.set_aspect("equal")
    axes[0].set_ylabel(r"Late-cooling change $d_2$")
    for axis in axes:
        axis.set_xlabel(r"Early-cooling change $d_1$")
    figure.suptitle("Same decision change · different definitions of size", y=0.97, color=INK, fontsize=14, weight="bold")
    figure.subplots_adjust(left=0.08, right=0.985, bottom=0.16, top=0.80, wspace=0.21)
    plt.show()
    plt.close(figure)
    return dict(l1=norm_values[0], l2=norm_values[1], linf=norm_values[2])


norms = show_norm_comparison()
print(norms)


### 5 · Level sets show equal scores; the gradient shows local change

A level set collects decisions with the same objective value:

> $\displaystyle \mathcal L_\alpha=\{x\in\mathcal X:f(x)=\alpha\}.$

On a contour plot, every curve is a two-dimensional level set. Moving along one curve keeps the plotted score fixed. Moving across curves changes it.

For a differentiable scalar objective, the gradient is

> $\displaystyle \nabla f(x)=\begin{bmatrix}\partial f/\partial x_1\\\partial f/\partial x_2\end{bmatrix}.$

The gradient is perpendicular to the local level set and points toward the fastest increase. Its negative points toward the fastest local decrease under the \(\ell_2\) norm. The figure evaluates the classroom score \(J(u;\lambda_E)=D(u)+\lambda_EE(u)\) at \(\lambda_E=1\). Purple identifies this fixed hyperparameter; it does not change the physical response of any fixed decision.

<div style="text-align: left; margin: 0.65rem 0 1.5rem 0;">
  <img src="https://raw.githubusercontent.com/sonamu-jun/system-design-and-optimization/main/02-2_mathematics_for_optimization/assets/04_level_sets_and_gradient.svg" alt="Contour map of the classroom score over early and late cooling with feasible region, a current candidate, the local gradient, and the negative-gradient descent direction" width="720" style="display: block; max-width: 100%; height: auto; margin: 0;">
</div>

This finite grid reveals the landscape at sampled points. It does not prove a continuous optimum. The finite-difference arrow is also a local numerical approximation.


In [ ]:
def show_level_sets_and_gradient(decision=(3.0, 2.0), energy_weight=1.0):
    plt = _pyplot()
    levels = np.linspace(MIN_COOLING, MAX_COOLING, 101)
    early_grid, late_grid = np.meshgrid(levels, levels)
    scores = np.empty_like(early_grid)
    feasible = np.empty_like(early_grid, dtype=bool)
    for row in range(len(levels)):
        for column in range(len(levels)):
            candidate = (early_grid[row, column], late_grid[row, column])
            scores[row, column] = score(candidate, energy_weight)
            feasible[row, column] = is_feasible(candidate)

    decision = np.asarray(decision, dtype=float)
    gradient = finite_difference_gradient(decision, energy_weight)
    gradient_direction = gradient / np.linalg.norm(gradient)
    figure, axis = plt.subplots(figsize=(8.0, 6.2))
    axis.contourf(early_grid, late_grid, feasible.astype(float), levels=[0.5, 1.5], colors=[TEAL], alpha=0.12)
    contours = axis.contour(early_grid, late_grid, scores, levels=[50, 75, 100, 150, 225, 325], cmap="Blues", linewidths=1.8)
    axis.clabel(contours, inline=True, fontsize=8, fmt=lambda value: f"J={value:g}")
    axis.scatter(*decision, s=150, color=ORANGE, edgecolor="white", linewidth=2, zorder=5, label="Current candidate")
    arrow_scale = 0.85
    axis.annotate(
        "", xy=decision + arrow_scale * gradient_direction, xytext=decision,
        arrowprops=dict(arrowstyle="-|>", color=BLUE, linewidth=3, mutation_scale=17),
    )
    axis.annotate(
        "", xy=decision - arrow_scale * gradient_direction, xytext=decision,
        arrowprops=dict(arrowstyle="-|>", color=TEAL, linewidth=3, mutation_scale=17),
    )
    axis.text(*(decision + 0.98 * gradient_direction), r"$\nabla J$", color=BLUE, fontsize=11, weight="bold")
    axis.text(*(decision - 1.18 * gradient_direction), r"$-\nabla J$", color=TEAL, fontsize=11, weight="bold")
    axis.text(0.97, 0.96, rf"$\lambda_E={energy_weight:g}$", transform=axis.transAxes, ha="right", va="top", color=PURPLE, fontsize=12, weight="bold", bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor=PURPLE, alpha=0.95))
    axis.plot([], [], color=TEAL, linewidth=8, alpha=0.18, label="Feasible region")
    axis.plot([], [], color=BLUE, linewidth=2.5, label=r"Gradient: fastest increase")
    axis.plot([], [], color=TEAL, linewidth=2.5, label=r"Negative gradient: local decrease")
    axis.set(
        xlim=(0, 5), ylim=(0, 5),
        xlabel=r"Early cooling $x_1$ (cooling units)",
        ylabel=r"Late cooling $x_2$ (cooling units)",
        title="Level sets turn objective values into a navigable landscape",
        xticks=np.arange(0, 6), yticks=np.arange(0, 6),
    )
    style_axis(axis)
    axis.legend(loc="lower right", fontsize=8, framealpha=0.94)
    axis.set_aspect("equal")
    figure.tight_layout()
    plt.show()
    plt.close(figure)
    return SimpleNamespace(gradient=gradient, scores=scores, feasible=feasible)


landscape = show_level_sets_and_gradient()
print(f"finite-difference gradient at x=[3, 2]^T: {landscape.gradient.round(3)}")


### 6 · Diagnose before choosing a search step

Use the objects in a fixed order:

| Question | Mathematical object | Classroom interpretation |
|:---|:---|:---|
| Where is the current decision? | Vector $x$ | Early and late cooling coordinates |
| How large is a proposed change? | Norm $\lVert d\rVert$ | Step size under a stated distance rule |
| Does the step align with increase? | Inner product $\nabla f^{\mathsf T}d$ | First-order sign of score change |
| Which decisions have equal score? | Level set $\mathcal L_\alpha$ | A contour of equal $J$ |
| Which local direction increases fastest? | Gradient $\nabla f$ | Local slope in decision space |

**Prediction.** If \(\nabla f(x)^{\mathsf T}d<0\), what score change do you expect for a sufficiently small positive step?

**Diagnosis.** A step has a small \(\ell_2\) norm but violates a cooling bound. Is it feasible? Explain why distance and feasibility answer different questions.

**Transfer.** If a binary ventilation choice is added, the decision becomes mixed. Which vector operations remain useful, and where does an ordinary gradient no longer describe every allowed move?

**Exit ticket.** Write one sentence that distinguishes the physical effect of changing \(x\) from the analytical effect of changing the norm or \(\lambda_E\).


### Takeaway

The reusable reasoning chain is

> **represent the decision with a vector \(x\) → measure a proposed change with \(\lVert d\rVert\) → predict directional change with \(\nabla f(x)^{\mathsf T}d\) → read equal values from level sets → use \(-\nabla f(x)\) as the steepest local \(\ell_2\)-descent direction**

A decision changes the real system. A norm and the energy weight \(\lambda_E\) are analyst settings. A gradient describes local objective change; it does not replace simulation, feasibility checking, or global search.
